# Kaggle model judging

Create this as a **utility benchmark** notebook with supported model access. Attach the supplied project zip and saved generation outputs. No GPU is required. This notebook has not been run against Kaggle model access yet.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile
REPO = Path('/kaggle/working/small-model-alignment')
if not REPO.exists():
    archives = list(Path('/kaggle/input').rglob('small-model-alignment.zip'))
    if len(archives) == 1:
        with zipfile.ZipFile(archives[0]) as archive:
            archive.extractall('/kaggle/working')
    else:
        sources = [p.parent for p in Path('/kaggle/input').rglob('submission_adapter.py')
                   if (p.parent / 'alignment/core.py').is_file()
                   and (p.parent / 'configs/sft.json').is_file()
                   and not (p.parent / 'outputs/sft/complete.json').exists()]
        if len(sources) != 1:
            raise RuntimeError('Attach exactly one project dataset containing the ZIP or unpacked project files')
        shutil.copytree(sources[0], REPO)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
def run(script, *arguments):
    subprocess.run([sys.executable, script, *map(str, arguments)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)


In [ ]:
import time
import kaggle_benchmarks as kbench
from scripts.judge_pairwise_kaggle import evaluate
pair_files = list(Path('/kaggle/input').rglob('uf_pairs.jsonl'))
if len(pair_files) != 1:
    raise RuntimeError('Attach exactly one generation dataset containing uf_pairs.jsonl')
PAIR_ROOT = pair_files[0].parent
JUDGE_MODEL = 'google/gemini-3.6-flash'
# If unavailable, choose another model supported in your Kaggle environment.
print('Judge:', JUDGE_MODEL, 'Pairs:', PAIR_ROOT)
@kbench.task(name='small_model_pairwise_evaluation')
def judge_track(llm, pairs_path: str, output_path: str, candidate: str, mode: str):
    return evaluate(pairs_path, output_path, candidate, JUDGE_MODEL, mode)


## Judge saved pairs

Five pairs per request, at least five seconds between attempts, up to three attempts, and checkpointed judgments. Rerunning reuses completed decisions. Separate tracks retain separate summaries.

In [ ]:
for filename, candidate, mode in [('uf', 'dpo-uf', 'general'), ('hh', 'dpo-hh', 'safety'), ('persona', 'persona-lora', 'persona')]:
    judge_track.run(llm=kbench.llms[JUDGE_MODEL], pairs_path=str(PAIR_ROOT / f'{filename}_pairs.jsonl'), output_path=str(REPO / f'outputs/judging/{filename}_judged.jsonl'), candidate=candidate, mode=mode)
    time.sleep(5)

## Inspect and preserve

Inspect summaries, unknown decisions, failure logs, and representative improvements/regressions. For persona, inspect style, usefulness, and safety separately. Save a notebook version. Use the actual training and evaluation files to write the final report, no more than eight pages.

In [ ]:
from alignment.io import read_rows
for path in sorted(Path('outputs/judging').glob('*.summary.json')):
    print(path.name, read_rows(path))